In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
import spacy
import nltk
import textstat
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
    BertTokenizer, BertModel, T5ForConditionalGeneration, T5Tokenizer,
    pipeline, BartForConditionalGeneration, BartTokenizer
)
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import gradio as gr
nlp = spacy.load('en_core_web_sm')
from datasets import load_dataset

In [ ]:
dataset = load_dataset("scientific_papers", "arxiv", split="train[:2%]")
print(f"Dataset loaded with {len(dataset)} examples")
print("\nSample Paper:")
print("Title:", dataset[0]['article'].split('\n')[0])
print("Abstract (first 200 chars):", dataset[0]['abstract'][:200] + "...")
print("Article length (chars):", len(dataset[0]['article']))

In [ ]:
def clean_text(text):
    text = re.sub(r'\\.*?{.*?}', '', text)
    text = re.sub(r'\$.*?\$', '[MATH]', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def preprocess_paper(paper, max_length=1024):
    lines = paper.strip().split('\n')
    title = lines[0] if lines else ""
    content = ' '.join(lines[1:])
    content = clean_text(content)
    if len(content) > max_length:
        content = content[:max_length]
    return {
        "title": title,
        "content": content
    }
processed_dataset = dataset.map(lambda x: {
    "processed_article": clean_text(x["article"]),
    "processed_abstract": clean_text(x["abstract"])
})
print("\nPreprocessed sample:")
print("Original abstract length:", len(dataset[0]['abstract']))
print("Processed abstract length:", len(processed_dataset[0]['processed_abstract']))

In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:

def analyze_vocabulary(texts, n=50):
    all_words = []
    stop_words = set(stopwords.words('english'))
    for text in texts[:500]:  
        words = word_tokenize(text.lower())
        words = [word for word in words if word.isalnum() and word not in stop_words]
        all_words.extend(words)
    word_counts = Counter(all_words)
    return word_counts.most_common(n)
common_words = analyze_vocabulary(processed_dataset['processed_abstract'])
print("\nMost common words in abstracts:")
for word, count in common_words:
    print(f"{word}: {count}")

In [ ]:
class TransformerSummarizer:
    def __init__(self, model_name="facebook/bart-large-cnn"):
        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        self.model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
        print(f"Loaded {model_name} model")
    def summarize(self, text, max_length=150, min_length=50):
        inputs = self.tokenizer.encode("summarize: " + text,
                                       return_tensors="pt",
                                       max_length=1024,
                                       truncation=True).to(device)
        summary_ids = self.model.generate(inputs,
                                         max_length=max_length,
                                         min_length=min_length,
                                         length_penalty=2.0,
                                         num_beams=4,
                                         early_stopping=True)
        summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary

In [ ]:
transformer_summarizer = TransformerSummarizer()
test_paper = processed_dataset[5]['processed_article']
test_summary = transformer_summarizer.summarize(test_paper[:1024])
print("\nTest Summary from BART:")
print(test_summary)

In [ ]:
class TextSimplifier:
    def __init__(self, model_name="t5-base"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
        print(f"Loaded {model_name} model for text simplification")
    def simplify(self, text, max_length=150):
        input_text = "simplify: " + text
        inputs = self.tokenizer.encode(input_text,
                                      return_tensors="pt",
                                      max_length=512,
                                      truncation=True).to(device)
        outputs = self.model.generate(inputs,
                                     max_length=max_length,
                                     min_length=30,
                                     length_penalty=2.0,
                                     num_beams=4,
                                     early_stopping=True)
        simplified_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return simplified_text
text_simplifier = TextSimplifier()

In [ ]:
complex_sentence = "The quantum mechanical model elucidates the probabilistic nature of electron behavior in atomic orbitals, highlighting the uncertainty principle's implications on our ability to simultaneously determine position and momentum."
simplified = text_simplifier.simplify(complex_sentence)
print("\nOriginal complex text:")
print(complex_sentence)
print("\nSimplified text:")
print(simplified)

In [ ]:
class BertExtractiveSummarizer:
    def __init__(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.model = BertModel.from_pretrained('bert-base-uncased').to(device)
        print("Loaded BERT for extractive summarization")
    def summarize(self, text, num_sentences=3):
        sentences = sent_tokenize(text)
        if len(sentences) <= num_sentences:
            return text
        inputs = self.tokenizer(sentences, return_tensors='pt',
                               padding=True, truncation=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = self.model(**inputs)
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        similarity_matrix = np.matmul(embeddings, embeddings.T)
        centrality_scores = np.sum(similarity_matrix, axis=1)
        top_sentence_indices = np.argsort(centrality_scores)[-num_sentences:]
        top_sentence_indices = sorted(top_sentence_indices)
        summary = ' '.join([sentences[i] for i in top_sentence_indices])
        return summary
bert_extractive = BertExtractiveSummarizer()

In [ ]:
extractive_summary = bert_extractive.summarize(test_paper[:2000])
print("\nBERT Extractive Summary:")
print(extractive_summary)

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
class KnowledgeEnhancer:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )
        self.knowledge_base = self._create_sample_knowledge_base()
        print("Initialized Knowledge Enhancer with sample knowledge base")
    def _create_sample_knowledge_base(self):
        knowledge_texts = [
            "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.",
            "Neural networks are computing systems inspired by the biological neural networks in animal brains.",
            "Natural language processing (NLP) is a subfield of linguistics, computer science, and AI concerned with interactions between computers and human language.",
            "Transformer models are a type of neural network architecture that uses self-attention mechanisms.",
            "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based machine learning technique for NLP pre-training developed by Google.",
            "T5 (Text-to-Text Transfer Transformer) treats every NLP problem as a text-to-text problem.",
            "BART (Bidirectional and Auto-Regressive Transformers) is a transformer encoder-decoder model designed for sequence-to-sequence tasks.",
            "Quantum mechanics is a fundamental theory in physics that describes nature at the scale of atoms and subatomic particles.",
            "LSTM (Long Short-Term Memory) is a type of recurrent neural network capable of learning long-term dependencies.",
            "RNN (Recurrent Neural Network) is a class of neural networks where connections between nodes form a directed graph along a temporal sequence."
        ]
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
        knowledge_docs = [{"content": text, "id": i} for i, text in enumerate(knowledge_texts)]
        vector_store = FAISS.from_texts(
            texts=[doc["content"] for doc in knowledge_docs],
            embedding=self.embeddings,
            metadatas=knowledge_docs
        )
        return vector_store
    def enhance_summary(self, summary, original_text, num_contexts=2):
        doc = nlp(original_text)
        key_terms = set()
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PRODUCT", "EVENT", "LAW", "WORK_OF_ART"]:
                key_terms.add(ent.text.lower())
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) <= 3:  
                key_terms.add(chunk.text.lower())
        retrieved_docs = self.knowledge_base.similarity_search(summary, k=num_contexts)
        enhanced_summary = summary + "\n\nAdditional explanations for beginners:\n"
        for i, doc in enumerate(retrieved_docs):
            enhanced_summary += f"{i+1}. {doc.page_content}\n"
        return enhanced_summary


In [ ]:
knowledge_enhancer = KnowledgeEnhancer()
beginner_summary = transformer_summarizer.summarize(test_paper[:1024], max_length=100)
enhanced_summary = knowledge_enhancer.enhance_summary(beginner_summary, test_paper[:1024])
print("\nRegular Summary:")
print(beginner_summary)
print("\nKnowledge-Enhanced Summary for Beginners:")
print(enhanced_summary)

In [ ]:
class AdaptiveSummarizer:    
    def __init__(self):
        self.transformer_summarizer = TransformerSummarizer()
        self.text_simplifier = TextSimplifier()
        self.bert_extractive = BertExtractiveSummarizer()
        self.knowledge_enhancer = KnowledgeEnhancer()
        print("Initialized Adaptive Summarizer System")
    def summarize(self, text, expertise_level="Intermediate", max_length=None):
        if max_length is None:
            max_length = 150 if expertise_level == "Expert" else 200
        if expertise_level == "Expert":
            base_summary = self.bert_extractive.summarize(text, num_sentences=5)
            key_findings = self.transformer_summarizer.summarize(
                text, max_length=100, min_length=50
            )
            final_summary = f"Technical Summary:\n{base_summary}\n\nKey Contributions:\n{key_findings}"
        elif expertise_level == "Intermediate":
            base_summary = self.transformer_summarizer.summarize(
                text, max_length=max_length, min_length=min(50, max_length // 2)
            )
            final_summary = base_summary
        else:  
            base_summary = self.transformer_summarizer.summarize(
                text, max_length=max_length // 2, min_length=min(30, max_length // 3)
            )
            simplified_summary = self.text_simplifier.simplify(base_summary)
            
            final_summary = self.knowledge_enhancer.enhance_summary(
                simplified_summary, text, num_contexts=3
            )
        return final_summary
    def analyze_readability(self, text): 
        metrics = {
            "Flesch Reading Ease": textstat.flesch_reading_ease(text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(text),
            "SMOG Index": textstat.smog_index(text),
            "Coleman-Liau Index": textstat.coleman_liau_index(text),
            "Automated Readability": textstat.automated_readability_index(text),
            "Dale-Chall Readability": textstat.dale_chall_readability_score(text)
        }
        return metrics
adaptive_summarizer = AdaptiveSummarizer()

In [ ]:
def evaluate_summary(generated_summary, reference_summary):
    result = {}
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(reference_summary, generated_summary)
    result["ROUGE-1"] = rouge_scores["rouge1"].fmeasure
    result["ROUGE-2"] = rouge_scores["rouge2"].fmeasure
    result["ROUGE-L"] = rouge_scores["rougeL"].fmeasure
    smoothie = SmoothingFunction().method1
    bleu_score = sentence_bleu(
        [reference_summary.split()],
        generated_summary.split(),
        smoothing_function=smoothie
    )
    result["BLEU"] = bleu_score
    P, R, F1 = bert_score([generated_summary], [reference_summary], lang='en', rescale_with_baseline=True)
    result["BERTScore"] = F1.item()
    result["Flesch-Kincaid Grade"] = textstat.flesch_kincaid_grade(generated_summary)
    result["Dale-Chall Score"] = textstat.dale_chall_readability_score(generated_summary)
    return result

In [ ]:
test_reference = processed_dataset[10]['processed_abstract']
test_paper_content = processed_dataset[10]['processed_article'][:2000]
summaries = {
    "Beginner": adaptive_summarizer.summarize(test_paper_content, "Beginner"),
    "Intermediate": adaptive_summarizer.summarize(test_paper_content, "Intermediate"),
    "Expert": adaptive_summarizer.summarize(test_paper_content, "Expert")
}
evaluation_results = {}
for level, summary in summaries.items():
    evaluation_results[level] = evaluate_summary(summary, test_reference)
print("\nEvaluation Results:")
for level, metrics in evaluation_results.items():
    print(f"\n--- {level} Level Summary ---")
    print(f"Summary: {summaries[level][:150]}...")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

In [ ]:
def plot_metrics(metrics_dict):   
    levels = list(metrics_dict.keys())
    metrics = list(metrics_dict[levels[0]].keys())
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, metric in enumerate(metrics[:6]):  
        values = [metrics_dict[level][metric] for level in levels]
        axes[i].bar(levels, values, color=['green', 'blue', 'red'])
        axes[i].set_title(metric)
        axes[i].set_ylim(0, max(values) * 1.2)  
        for j, value in enumerate(values):
            axes[i].text(j, value + (max(values) * 0.05), f'{value:.3f}',
                        ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.savefig('metrics_comparison.png')
    plt.show()
plot_metrics(evaluation_results)

In [ ]:
def create_interactive_demo():
    input_box = widgets.Textarea(
        placeholder='Paste your scientific paper here...',
        description='Input:',
        layout=widgets.Layout(width='100%', height='200px')
    )
    expertise_dropdown = widgets.Dropdown(
        options=['Beginner', 'Intermediate', 'Expert'],
        value='Intermediate',
        description='Reader Expertise:'
    )
    max_length_slider = widgets.IntSlider(
        value=200,
        min=100,
        max=500,
        step=50,
        description='Max Length:',
        disabled=False
    )
    summarize_button = widgets.Button(
        description='Generate Summary',
        button_style='success',
        tooltip='Click to generate a summary based on expertise level'
    )
    output_area = widgets.Output()
    metrics_output = widgets.Output()
    def on_button_clicked(b):
        with output_area:
            clear_output()
            text = input_box.value

            if not text or len(text) < 100:
                print("Please enter a longer scientific text (at least 100 characters).")
                return
            expertise = expertise_dropdown.value
            max_length = max_length_slider.value
            print(f"Generating {expertise.lower()}-level summary...")
            summary = adaptive_summarizer.summarize(text, expertise, max_length)
            print("\n--- Summary ---\n")
            print(summary)
        with metrics_output:
            clear_output()
            readability_metrics = adaptive_summarizer.analyze_readability(summary)
            print("--- Readability Metrics ---")
            for metric, value in readability_metrics.items():
                print(f"{metric}: {value:.2f}")
    summarize_button.on_click(on_button_clicked)
    input_section = widgets.VBox([
        widgets.HTML("<h3>Adaptive Scientific Paper Summarization</h3>"),
        input_box
    ])
    controls = widgets.HBox([
        expertise_dropdown,
        max_length_slider,
        summarize_button
    ])
    output_section = widgets.VBox([
        widgets.HTML("<h4>Generated Summary</h4>"),
        output_area,
        widgets.HTML("<h4>Metrics</h4>"),
        metrics_output
    ])
    return widgets.VBox([input_section, controls, output_section])
interactive_demo = create_interactive_demo()
display(interactive_demo)

In [ ]:
def create_gradio_interface():
    def summarize_paper(paper_text, expertise_level, max_length):
        if len(paper_text) < 100:
            return "Please enter a longer scientific text (at least 100 characters)."
        summary = adaptive_summarizer.summarize(
            paper_text, expertise_level, int(max_length)
        )
        readability_metrics = adaptive_summarizer.analyze_readability(summary)
        metrics_text = "\n\n--- Readability Metrics ---\n"
        for metric, value in readability_metrics.items():
            metrics_text += f"{metric}: {value:.2f}\n"
        return summary + metrics_text
    iface = gr.Interface(
        fn=summarize_paper,
        inputs=[
            gr.Textbox(lines=10, placeholder="Paste scientific paper text here...", label="Paper Text"),
            gr.Radio(["Beginner", "Intermediate", "Expert"], label="Reader Expertise", value="Intermediate"),
            gr.Slider(100, 500, value=200, step=50, label="Maximum Summary Length")
        ],
        outputs=gr.Textbox(label="Generated Summary"),
        title="Adaptive Scientific Paper Summarizer",
        description="This tool generates summaries of scientific papers customized to different levels of expertise.",
        examples=[
            [processed_dataset[15]["processed_article"][:2000], "Beginner", 200],
            [processed_dataset[20]["processed_article"][:2000], "Intermediate", 200],
            [processed_dataset[25]["processed_article"][:2000], "Expert", 200]
        ]
    )
    return iface
gradio_interface = create_gradio_interface()
gradio_interface.launch(share=True)

In [ ]:
def demonstrate_full_system():
    print("==== Adaptive Summarization System Demonstration ====\n")
    test_index = 30
    paper = processed_dataset[test_index]['processed_article'][:3000]
    reference = processed_dataset[test_index]['processed_abstract']
    print(f"Original Paper (first 300 chars):\n{paper[:300]}...\n")
    print(f"Original Abstract:\n{reference}\n")
    print("Generating summaries for different expertise levels...")
    summaries = {}
    for level in ["Beginner", "Intermediate", "Expert"]:
        summary = adaptive_summarizer.summarize(paper, level)
        summaries[level] = summary
        print(f"\n--- {level} Level Summary ---\n")
        print(summary)
        metrics = evaluate_summary(summary, reference)
        print("\nMetrics:")
        for metric, value in metrics.items():
            print(f"{metric}: {value:.4f}")
    print("\n--- Readability Comparison ---")
    readability_scores = {}
    for level, summary in summaries.items():
        fk_grade = textstat.flesch_kincaid_grade(summary)
        reading_ease = textstat.flesch_reading_ease(summary)
        readability_scores[level] = {"FK Grade": fk_grade, "Reading Ease": reading_ease}
        print(f"{level}: FK Grade = {fk_grade:.2f}, Reading Ease = {reading_ease:.2f}")
    levels = list(readability_scores.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fk_values = [readability_scores[level]["FK Grade"] for level in levels]
    ax1.bar(levels, fk_values, color=['green', 'blue', 'red'])
    ax1.set_title("Flesch-Kincaid Grade Level")
    ax1.set_ylabel("Grade Level")
    re_values = [readability_scores[level]["Reading Ease"] for level in levels]
    ax2.bar(levels, re_values, color=['green', 'blue', 'red'])
    ax2.set_title("Flesch Reading Ease")
    ax2.set_ylabel("Score")
    plt.tight_layout()
    plt.savefig('readability_comparison.png')
    plt.show()
    return summaries
demo_summaries = demonstrate_full_system()

In [22]:
import wikipedia
from langchain_community.document_loaders import WikipediaLoader
from langchain.retrievers import BM25Retriever
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import requests
from tenacity import retry, stop_after_attempt, wait_exponential
import json

In [ ]:
class EnhancedKnowledgeEnhancer:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )
        self.knowledge_base = self._create_enhanced_knowledge_base()
        print("Initialized Enhanced Knowledge Enhancer with Wikipedia integration")
        wikipedia.set_lang("en")
    def _create_enhanced_knowledge_base(self):
        knowledge_texts = [
            "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.",
            "Neural networks are computing systems inspired by the biological neural networks in animal brains.",
            "Natural language processing (NLP) is a subfield of linguistics, computer science, and AI concerned with interactions between computers and human language.",
            "Transformer models are a type of neural network architecture that uses self-attention mechanisms.",
            "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based machine learning technique for NLP pre-training developed by Google.",
            "T5 (Text-to-Text Transfer Transformer) treats every NLP problem as a text-to-text problem.",
            "BART (Bidirectional and Auto-Regressive Transformers) is a transformer encoder-decoder model designed for sequence-to-sequence tasks.",
            "Quantum mechanics is a fundamental theory in physics that describes nature at the scale of atoms and subatomic particles.",
            "LSTM (Long Short-Term Memory) is a type of recurrent neural network capable of learning long-term dependencies.",
            "RNN (Recurrent Neural Network) is a class of neural networks where connections between nodes form a directed graph along a temporal sequence.",
            "Reinforcement learning is an area of machine learning concerned with how intelligent agents ought to take actions in an environment to maximize cumulative reward.",
            "Computer vision is a field of artificial intelligence that trains computers to interpret and understand the visual world.",
            "Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning.",
            "Unsupervised learning is a type of machine learning where models are trained using data that is neither classified nor labeled.",
            "Transfer learning is a machine learning technique where a model developed for one task is reused as the starting point for a model on a second task.",
            "Attention mechanisms in neural networks allow the model to focus on specific parts of the input sequence when generating an output.",
            "Self-attention, also known as intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence.",
            "Fine-tuning is the process of taking a pre-trained model and further training it on a specific dataset for a particular task.",
            "A language model is a probability distribution over sequences of words or tokens that is used to generate text or predict the next word in a sequence."
        ]
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
        knowledge_docs = [{"content": text, "id": i} for i, text in enumerate(knowledge_texts)]
        vector_store = FAISS.from_texts(
            texts=[doc["content"] for doc in knowledge_docs],
            embedding=self.embeddings,
            metadatas=knowledge_docs
        )
        return vector_store
    def _extract_key_terms(self, text, max_terms=5):
        doc = nlp(text)
        key_terms = []
        term_scores = {}
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PRODUCT", "EVENT", "LAW", "WORK_OF_ART"]:
                term = ent.text.lower()
                if term not in term_scores:
                    term_scores[term] = 1
                else:
                    term_scores[term] += 1
        for chunk in doc.noun_chunks:
            if 1 <= len(chunk.text.split()) <= 3:  
                term = chunk.text.lower()
                if any(token.pos_ == "ADJ" for token in chunk):
                    term_scores[term] = term_scores.get(term, 0) + 1.5
                else:
                    term_scores[term] = term_scores.get(term, 0) + 0.8
        sorted_terms = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)
        return [term for term, score in sorted_terms[:max_terms]]
    def _get_wikipedia_content(self, term, max_sentences=3):
        try:
            search_results = wikipedia.search(term, results=1)
            if not search_results:
                return None
            page = wikipedia.page(search_results[0], auto_suggest=False)
            summary = page.summary
            sentences = sent_tokenize(summary)
            return ' '.join(sentences[:max_sentences])
        except (wikipedia.exceptions.DisambiguationError, wikipedia.exceptions.PageError) as e:
            print(f"Wikipedia error for term '{term}': {str(e)}")
            return None
        except Exception as e:
            print(f"Error retrieving Wikipedia content for '{term}': {str(e)}")
            return None
    def enhance_summary(self, summary, original_text, num_contexts=3):
        key_terms = self._extract_key_terms(original_text)
        retrieved_docs = self.knowledge_base.similarity_search(summary, k=num_contexts)
        enhanced_summary = summary + "\n\nAdditional explanations for beginners:\n"
        for i, doc in enumerate(retrieved_docs):
            enhanced_summary += f"{i+1}. {doc.page_content}\n"
        wiki_explanations = []
        for term in key_terms:
            wiki_content = self._get_wikipedia_content(term)
            if wiki_content:
                wiki_explanations.append(f"* {term.capitalize()}: {wiki_content}")
        if wiki_explanations:
            enhanced_summary += "\n\nKey concepts from this paper:\n"
            enhanced_summary += "\n\n".join(wiki_explanations)
        return enhanced_summary

In [ ]:
class KnowledgeGraphEnhancer:
    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
        print("Initialized Knowledge Graph Enhancer")
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _query_wikidata(self, entity_name):
        endpoint_url = "https://query.wikidata.org/sparql"
        query = f"""
        SELECT ?item ?itemLabel ?itemDescription ?field ?fieldLabel
        WHERE {{
            ?item rdfs:label "{entity_name}"@en.
            OPTIONAL {{ ?item wdt:P101 ?field. }}
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        LIMIT 5
        """
        try:
            r = requests.get(endpoint_url, params={'format': 'json', 'query': query})
            data = r.json()
            results = data.get('results', {}).get('bindings', [])
            if not results:
                return None
            entity_info = {
                "description": results[0].get('itemDescription', {}).get('value', ''),
                "fields": []
            }
            for result in results:
                if 'fieldLabel' in result:
                    field = result['fieldLabel']['value']
                    if field not in entity_info["fields"]:
                        entity_info["fields"].append(field)
            return entity_info
        except Exception as e:
            print(f"Error querying Wikidata: {str(e)}")
            return None
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _query_semantic_scholar(self, query):
        try:
            url = f"https://api.semanticscholar.org/graph/v1/paper/search?query={query}&limit=3&fields=title,abstract,year,authors,url"
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                papers = data.get('data', [])
                return papers
            else:
                print(f"Error querying Semantic Scholar: {response.status_code}")
                return []
        except Exception as e:
            print(f"Error in Semantic Scholar query: {str(e)}")
            return []
    def enhance_expert_summary(self, summary, original_text):
        doc = nlp(original_text)
        key_entities = set()
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "WORK_OF_ART", "EVENT", "LAW"]:
                key_entities.add(ent.text)
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) >= 2 and len(chunk.text.split()) <= 4:
                key_entities.add(chunk.text)
        entity_counter = Counter([e.lower() for e in key_entities])
        top_entities = [e for e, _ in entity_counter.most_common(3)]
        kg_info = ""
        for entity in top_entities:
            entity_info = self._query_wikidata(entity)
            if entity_info:
                kg_info += f"\n\n### {entity.capitalize()}\n"
                kg_info += f"Description: {entity_info['description']}\n"
                if entity_info['fields']:
                    kg_info += f"Fields: {', '.join(entity_info['fields'])}\n"
            papers = self._query_semantic_scholar(entity)
            if papers:
                kg_info += f"\nRecent research on {entity}:\n"
                for i, paper in enumerate(papers[:2], 1):
                    kg_info += f"{i}. {paper.get('title', 'Untitled')} ({paper.get('year', 'N/A')})\n"
                    authors = ", ".join([a.get('name', '') for a in paper.get('authors', [])][:3])
                    if authors:
                        kg_info += f"   Authors: {authors}\n"
        if kg_info:
            enhanced_summary = summary + "\n\n## Related Knowledge Graph Information" + kg_info
            return enhanced_summary
        else:
            return summary

In [ ]:
class ControlledTransformerSummarizer:
    def __init__(self, model_name="facebook/bart-large-cnn"):
        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        self.model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
        print(f"Loaded {model_name} model with control token capabilities")
        self.control_tokens = {
            "Beginner": "[BEGINNER]",
            "Intermediate": "[INTERMEDIATE]",
            "Expert": "[EXPERT]"
        }
    def summarize(self, text, expertise_level="Intermediate", max_length=150, min_length=50):
        control_token = self.control_tokens.get(expertise_level, self.control_tokens["Intermediate"])
        controlled_text = f"{control_token} {text}"
        inputs = self.tokenizer.encode(
            "summarize: " + controlled_text,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        ).to(device)
        if expertise_level == "Beginner":
            summary_ids = self.model.generate(
                inputs,
                max_length=max_length,
                min_length=min_length,
                length_penalty=2.0,
                num_beams=4,
                repetition_penalty=1.3,
                early_stopping=True
            )
        elif expertise_level == "Expert":
            summary_ids = self.model.generate(
                inputs,
                max_length=max_length,
                min_length=min_length,
                length_penalty=1.0,
                num_beams=5,
                repetition_penalty=1.1,
                early_stopping=True
            )
        else:
            summary_ids = self.model.generate(
                inputs,
                max_length=max_length,
                min_length=min_length,
                length_penalty=2.0,
                num_beams=4,
                early_stopping=True
            )
        summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary

In [ ]:
import requests
import json
import os
import re
from tenacity import retry, stop_after_attempt, wait_exponential
import groq
class LLMEvaluator:
    def __init__(self, api_key=None):
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        if self.api_key:
            os.environ["GROQ_API_KEY"] = self.api_key
            try:
                self.client = groq.Client(api_key=self.api_key)
                print("Groq client initialized successfully")
            except Exception as e:
                print(f"Error initializing Groq client: {e}")
                print("LLM evaluation will be simulated")
                self.client = None
        else:
            print("No Groq API key provided. LLM evaluation will be simulated.")
            self.client = None
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _call_groq_api(self, prompt, model="llama3-70b-8192"):
        if self.client is None:
            print("Simulating Groq API call (no client available)")
            return {"success": True, "response": self._simulate_llm_response()}
        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,
                max_tokens=1000
            )
            return {
                "success": True,
                "response": response.choices[0].message.content
            }
        except Exception as e:
            print(f"Exception during API call: {str(e)}")
            return {"success": False, "error": str(e)}
    def _simulate_llm_response(self):
        return """
        {
            "content_accuracy": 7,
            "technical_correctness": 8,
            "readability": 8,
            "appropriate_complexity": 7,
            "overall_quality": 7.5,
            "strengths": "The summary captures the main points of the original text well. It uses appropriate language for the target audience and maintains good coherence.",
            "weaknesses": "Some technical details are simplified too much, potentially losing nuance. The summary could better explain the significance of the findings.",
            "suggestions": "Consider adding a brief explanation of why this research matters and its potential applications."
        }
        """
    def evaluate_summary(self, original_text, summary, reference_summary=None, expertise_level="Intermediate"):
        prompt = f"""
        You are an expert evaluator of scientific summaries. Please evaluate the following summary
        of a scientific text based on the following criteria:
        1. Clarity: How clear and understandable is the summary?
        2. Accuracy: How accurately does it capture the key information from the original text?
        3. Appropriateness: How appropriate is it for a reader with {expertise_level.lower()} expertise?
        4. Overall Quality: What is the overall quality of the summary?
        For each criterion, provide a score from 1-5 (where 5 is best) and a brief explanation.
        Original Text:
        {original_text}
        Summary (for {expertise_level} level):
        {summary}
        """
        if reference_summary:
            prompt += f"\nReference summary (for comparison):\n{reference_summary}"
        prompt += "\nProvide your evaluation in JSON format with these keys:\nscores (with clarity, accuracy, appropriateness, overall) and feedback."
        result = self._call_groq_api(prompt)
        if result["success"]:
            try:
                content = result["response"]
                json_match = re.search(r'```json\n(.*?)\n```', content, re.DOTALL)
                if json_match:
                    json_str = json_match.group(1)
                else:
                    json_match = re.search(r'\{.*\}', content, re.DOTALL)
                    if json_match:
                        json_str = json_match.group(0)
                    else:
                        json_str = content
                try:
                    evaluation = json.loads(json_str)
                    return evaluation
                except json.JSONDecodeError:
                    return {
                        'scores': {
                            'clarity': 3.5,
                            'accuracy': 3.5,
                            'appropriateness': 3.5,
                            'overall': 3.5
                        },
                        'feedback': f"Unable to parse LLM response. Raw response: {content[:100]}..."
                    }
            except Exception as e:
                print(f"Error processing LLM response: {e}")
                return {
                    'scores': {
                        'clarity': 3.0,
                        'accuracy': 3.0,
                        'appropriateness': 3.0,
                        'overall': 3.0
                    },
                    'feedback': f"Error occurred during response processing: {str(e)}"
                }
        else:
            print(f"Error in LLM evaluation: {result.get('error', 'Unknown error')}")
            return {
                'scores': {
                    'clarity': 0,
                    'accuracy': 0,
                    'appropriateness': 0,
                    'overall': 0
                },
                'feedback': f"API error: {result.get('error', 'Unknown error')}"
            }

In [ ]:
class EnhancedAdaptiveSummarizer:
    def __init__(self):
        self.controlled_summarizer = ControlledTransformerSummarizer()
        self.text_simplifier = TextSimplifier()
        self.bert_extractive = BertExtractiveSummarizer()
        self.enhanced_knowledge = EnhancedKnowledgeEnhancer()
        self.kg_enhancer = KnowledgeGraphEnhancer()
        self.llm_evaluator = LLMEvaluator(api_key="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")
        print("Initialized Enhanced Adaptive Summarizer System with all components")
    def summarize(self, text, expertise_level="Intermediate", max_length=None, evaluate=False):
        if max_length is None:
            max_length = 150 if expertise_level == "Expert" else 200

        base_summary = self.controlled_summarizer.summarize(
            text,
            expertise_level=expertise_level,
            max_length=max_length,
            min_length=min(50, max_length // 2)
        )
        if expertise_level == "Expert":
            extractive_summary = self.bert_extractive.summarize(text, num_sentences=5)
            key_findings = base_summary
            combined_summary = f"Technical Summary:\n{extractive_summary}\n\nKey Contributions:\n{key_findings}"
            final_summary = self.kg_enhancer.enhance_expert_summary(combined_summary, text)
        elif expertise_level == "Intermediate":
            final_summary = base_summary
        else:
            shorter_summary = self.controlled_summarizer.summarize(
                text,
                expertise_level="Beginner",
                max_length=max_length // 2,
                min_length=min(30, max_length // 3)
            )
            simplified_summary = self.text_simplifier.simplify(shorter_summary)
            final_summary = self.enhanced_knowledge.enhance_summary(simplified_summary, text, num_contexts=3)
        if evaluate:
            evaluation = self.llm_evaluator.evaluate_summary(
                original_text=text,
                summary=final_summary,
                expertise_level=expertise_level
            )
            return final_summary, evaluation
        else:
            return final_summary
    def analyze_readability(self, text):
        metrics = {
            "Flesch Reading Ease": textstat.flesch_reading_ease(text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(text),
            "SMOG Index": textstat.smog_index(text),
            "Coleman-Liau Index": textstat.coleman_liau_index(text),
            "Automated Readability": textstat.automated_readability_index(text),
            "Dale-Chall Readability": textstat.dale_chall_readability_score(text)
        }
        return metrics

In [ ]:
def create_enhanced_gradio_interface():
    enhanced_summarizer = EnhancedAdaptiveSummarizer()
    def summarize_paper(paper_text, expertise_level, max_length, perform_evaluation):
        if len(paper_text) < 100:
            return "Please enter a longer scientific text (at least 100 characters).", None
        try:
            if perform_evaluation:
                summary, evaluation = enhanced_summarizer.summarize(
                    paper_text, expertise_level, int(max_length), evaluate=True
                )
                if isinstance(evaluation, dict) and 'scores' in evaluation:
                    scores = evaluation.get('scores', {})
                    eval_text = f"""
                    ## Evaluation Results
                    **Clarity**: {scores.get('clarity', 'N/A')}/5  
                    **Accuracy**: {scores.get('accuracy', 'N/A')}/5  
                    **Appropriateness**: {scores.get('appropriateness', 'N/A')}/5  
                    **Overall Quality**: {scores.get('overall', 'N/A')}/5  
                    **Feedback**: {evaluation.get('feedback', 'No feedback provided')}
                    """
                else:
                    eval_text = f"""
                    ## Evaluation Results

                    {str(evaluation)}
                    """
                readability_metrics = enhanced_summarizer.analyze_readability(summary)
                metrics_text = "\n\n## Readability Metrics\n"
                for metric, value in readability_metrics.items():
                    metrics_text += f"**{metric}**: {value:.2f}\n"
                return summary, eval_text + metrics_text
            else:
                summary = enhanced_summarizer.summarize(
                    paper_text, expertise_level, int(max_length), evaluate=False
                )
                readability_metrics = enhanced_summarizer.analyze_readability(summary)
                metrics_text = "\n\n## Readability Metrics\n"
                for metric, value in readability_metrics.items():
                    metrics_text += f"**{metric}**: {value:.2f}\n"

                return summary, metrics_text
        except Exception as e:
            import traceback
            error_details = traceback.format_exc()
            return f"Error generating summary: {str(e)}\n\nDetails: {error_details}", None
    iface = gr.Interface(
        fn=summarize_paper,
        inputs=[
            gr.Textbox(
                lines=10,
                placeholder="Paste scientific paper text here...",
                label="Paper Text"
            ),
            gr.Radio(
                ["Beginner", "Intermediate", "Expert"],
                label="Reader Expertise",
                value="Intermediate"
            ),
            gr.Slider(
                100, 500,
                value=200,
                step=50,
                label="Maximum Summary Length"
            ),
            gr.Checkbox(
                label="Perform LLM-based Evaluation",
                value=False,
                info="Evaluates summary quality using LLM (may take longer)"
            )
        ],
        outputs=[
            gr.Textbox(label="Generated Summary"),
            gr.Markdown(label="Evaluation Results")
        ],
        title="Enhanced Adaptive Scientific Paper Summarizer",
        description="""This tool generates summaries of scientific papers customized to different levels of expertise.
- **Beginner**: Simplified language with additional explanations of key concepts  
- **Intermediate**: Standard summary balancing technical accuracy and readability  
- **Expert**: Technical summary with research context and knowledge graph enhancements
The LLM evaluation option provides detailed feedback on summary quality.""",
        examples=[
            [processed_dataset[15]["processed_article"][:2000], "Beginner", 200, False],
            [processed_dataset[20]["processed_article"][:2000], "Intermediate", 200, False],
            [processed_dataset[25]["processed_article"][:2000], "Expert", 200, True]
        ],
        allow_flagging="never"
    )
    return iface
enhanced_gradio_interface = create_enhanced_gradio_interface()
enhanced_gradio_interface.launch(share=True)

In [ ]:
def demonstrate_enhanced_system():
    print("==== Enhanced Adaptive Summarization System Demonstration ====\n")
    test_index = 30
    paper = processed_dataset[test_index]['processed_article'][:3000]
    reference = processed_dataset[test_index]['processed_abstract']
    print(f"Original Paper (first 300 chars):\n{paper[:300]}...\n")
    print(f"Original Abstract:\n{reference}\n")
    enhanced_summarizer = EnhancedAdaptiveSummarizer()
    print("Generating summaries for different expertise levels...")
    summaries = {}
    evaluations = {}
    for level in ["Beginner", "Intermediate", "Expert"]:
        print(f"\nProcessing {level} level summary...")
        summary, evaluation = enhanced_summarizer.summarize(paper, level, evaluate=True)
        summaries[level] = summary
        evaluations[level] = evaluation
        print(f"\n--- {level} Level Summary ---\n")
        print(summary)
        print("\n--- Evaluation Results ---")
        if isinstance(evaluation, dict) and 'scores' in evaluation:
            scores = evaluation.get('scores', {})
            print(f"Clarity: {scores.get('clarity', 'N/A')}/5")
            print(f"Accuracy: {scores.get('accuracy', 'N/A')}/5")
            print(f"Appropriateness: {scores.get('appropriateness', 'N/A')}/5")
            print(f"Overall Quality: {scores.get('overall', 'N/A')}/5")
            print(f"\nFeedback: {evaluation.get('feedback', 'No feedback provided')}")
        else:
            print("Evaluation data structure not as expected:")
            print(evaluation)
    print("\n--- Readability Comparison ---")
    readability_scores = {}
    for level, summary in summaries.items():
        fk_grade = textstat.flesch_kincaid_grade(summary)
        reading_ease = textstat.flesch_reading_ease(summary)
        readability_scores[level] = {"FK Grade": fk_grade, "Reading Ease": reading_ease}
        print(f"{level}: FK Grade = {fk_grade:.2f}, Reading Ease = {reading_ease:.2f}")
    levels = list(readability_scores.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fk_values = [readability_scores[level]["FK Grade"] for level in levels]
    ax1.bar(levels, fk_values, color=['green', 'blue', 'red'])
    ax1.set_title("Flesch-Kincaid Grade Level")
    ax1.set_ylabel("Grade Level")
    re_values = [readability_scores[level]["Reading Ease"] for level in levels]
    ax2.bar(levels, re_values, color=['green', 'blue', 'red'])
    ax2.set_title("Flesch Reading Ease")
    ax2.set_ylabel("Score")
    plt.tight_layout()
    plt.savefig('enhanced_readability_comparison.png')
    plt.show()
    fig, ax = plt.subplots(figsize=(14, 7))
    metrics = ["clarity", "accuracy", "appropriateness", "overall"]
    metric_labels = ["Clarity", "Accuracy", "Appropriateness", "Overall\nQuality"]
    x = np.arange(len(metrics))
    width = 0.25
    for i, level in enumerate(levels):
        if isinstance(evaluations[level], dict) and 'scores' in evaluations[level]:
            scores = evaluations[level].get('scores', {})
            values = [scores.get(metric, 0) for metric in metrics]
        else:
            values = [0 for _ in metrics]
        ax.bar(x + (i-1)*width, values, width, label=level, color=["green", "blue", "red"][i])
    ax.set_ylabel('Score (out of 5)')
    ax.set_title('LLM Evaluation Results by Expertise Level')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.legend()
    ax.set_ylim(0, 5)
    for i, level in enumerate(levels):
        if isinstance(evaluations[level], dict) and 'scores' in evaluations[level]:
            scores = evaluations[level].get('scores', {})
            values = [scores.get(metric, 0) for metric in metrics]
        else:
            values = [0 for _ in metrics]
        for j, v in enumerate(values):
            ax.text(j + (i-1)*width, v + 0.15, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig('enhanced_evaluation_comparison.png')
    plt.show()
    return summaries, evaluations
enhanced_summaries, enhanced_evaluations = demonstrate_enhanced_system()

In [ ]:
def run_comparative_evaluation():
    print("==== Comparative Evaluation: Original vs Enhanced Systems ====\n")
    test_indices = [10, 20, 30, 40, 50]
    original_summarizer = AdaptiveSummarizer()
    enhanced_summarizer = EnhancedAdaptiveSummarizer()
    results = {
        "Original": {level: {"ROUGE-1": [], "ROUGE-2": [], "ROUGE-L": [], "BLEU": [], "BERTScore": [], "FK Grade": [], "Reading Ease": []} for level in ["Beginner", "Intermediate", "Expert"]},
        "Enhanced": {level: {"ROUGE-1": [], "ROUGE-2": [], "ROUGE-L": [], "BLEU": [], "BERTScore": [], "FK Grade": [], "Reading Ease": [], "LLM Overall": []} for level in ["Beginner", "Intermediate", "Expert"]}
    }
    for i, idx in enumerate(test_indices):
        print(f"\nProcessing test sample {i+1}/{len(test_indices)} (dataset index {idx})...")
        paper = processed_dataset[idx]['processed_article'][:3000]
        reference = processed_dataset[idx]['processed_abstract']
        for level in ["Beginner", "Intermediate", "Expert"]:
            print(f"  Generating {level} summaries...")
            original_summary = original_summarizer.summarize(paper, level)
            enhanced_summary, evaluation = enhanced_summarizer.summarize(paper, level, evaluate=True)
            orig_metrics = evaluate_summary(original_summary, reference)
            enhanced_metrics = evaluate_summary(enhanced_summary, reference)
            for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "BERTScore"]:
                results["Original"][level][metric].append(orig_metrics[metric])
                results["Enhanced"][level][metric].append(enhanced_metrics[metric])
            results["Original"][level]["FK Grade"].append(textstat.flesch_kincaid_grade(original_summary))
            results["Original"][level]["Reading Ease"].append(textstat.flesch_reading_ease(original_summary))
            results["Enhanced"][level]["FK Grade"].append(textstat.flesch_kincaid_grade(enhanced_summary))
            results["Enhanced"][level]["Reading Ease"].append(textstat.flesch_reading_ease(enhanced_summary))
            if isinstance(evaluation, dict) and 'scores' in evaluation:
                llm_overall_score = evaluation['scores'].get('overall', 0)
            else:
                llm_overall_score = 0
            results["Enhanced"][level]["LLM Overall"].append(llm_overall_score)

    averages = {
        "Original": {level: {} for level in ["Beginner", "Intermediate", "Expert"]},
        "Enhanced": {level: {} for level in ["Beginner", "Intermediate", "Expert"]}
    }
    for system in ["Original", "Enhanced"]:
        for level in ["Beginner", "Intermediate", "Expert"]:
            for metric, values in results[system][level].items():
                averages[system][level][metric] = sum(values) / len(values)
    print("\n==== Average Evaluation Results ====")
    for level in ["Beginner", "Intermediate", "Expert"]:
        print(f"\n--- {level} Level ---")
        print(f"{'Metric':<20} {'Original':<10} {'Enhanced':<10} {'Diff':<10}")
        print("-" * 50)
        for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "BERTScore", "FK Grade", "Reading Ease"]:
            orig_val = averages["Original"][level][metric]
            enhanced_val = averages["Enhanced"][level][metric]
            diff = enhanced_val - orig_val
            if metric == "FK Grade":
                if level == "Beginner":
                    better = "+" if diff < 0 else "-"
                elif level == "Expert":
                    better = "+" if diff > 0 else "-"
                else:
                    better = "+" if abs(diff) < 1 else "-"
            elif metric == "Reading Ease":
                if level == "Beginner":
                    better = "+" if diff > 0 else "-"
                elif level == "Expert":
                    better = "+" if diff < 0 else "-"
                else:
                    better = "+" if abs(diff) < 5 else "-"
            else:
                better = "+" if diff > 0 else "-"
            print(f"{metric:<20} {orig_val:.4f}     {enhanced_val:.4f}     {diff:.4f} {better}")
        if "LLM Overall" in averages["Enhanced"][level]:
            print(f"\nLLM Overall Quality Score: {averages['Enhanced'][level]['LLM Overall']:.2f}/5")
    plot_comparative_results(averages)
    return results, averages
def plot_comparative_results(averages):
    metrics_to_plot = ["ROUGE-1", "ROUGE-L", "BERTScore", "Reading Ease"]
    levels = ["Beginner", "Intermediate", "Expert"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()
    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        x = np.arange(len(levels))
        width = 0.35
        orig_values = [averages["Original"][level][metric] for level in levels]
        enhanced_values = [averages["Enhanced"][level][metric] for level in levels]
        rects1 = ax.bar(x - width/2, orig_values, width, label='Original')
        rects2 = ax.bar(x + width/2, enhanced_values, width, label='Enhanced')
        ax.set_ylabel('Score')
        ax.set_title(f'{metric} Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(levels)
        ax.legend()
        for rect in rects1:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8)
        for rect in rects2:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    plt.savefig('comparative_results.png')
    plt.show()
    llm_scores = [averages["Enhanced"][level]["LLM Overall"] for level in levels]
    bars = plt.bar(levels, llm_scores, color=['green', 'blue', 'red'])
    plt.ylabel('LLM Overall Quality Score (out of 5)')
    plt.title('LLM Evaluation of Enhanced Summaries')
    plt.ylim(0, 5)
    for bar in bars:
        height = bar.get_height()
        plt.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                     xytext=(0, 3), textcoords="offset points",
                     ha='center', va='bottom')
    plt.savefig('llm_evaluation_scores.png')
    plt.show()
comparative_results, comparative_averages = run_comparative_evaluation()